# PKG — Counterparty Locatability  ·  v2
### Can we infer where an external account is, from who it pays?
**PNC Treasury Management · Data Science**

---

## What changed after the first full run

The v1 run produced 5.49M degree-1 experiments and a working estimator, but
five defects suppressed every headline number. All are fixed here.

| # | Defect found in the v1 run | Fix |
|---|---|---|
| 1 | **`bandwidth = 0` degeneracy.** When the leave-one-out leaves a neighbour with a single counterparty, `R̄' = 1` exactly and the bandwidth collapses to 0 — maximally *confident* and 300 km wrong. 1.17M pairs, 21% of the census, sat at the confident end of the feature | `kc_S ≥ MIN_CP_BW` guard; bandwidth is **NaN below 2**, untrusted below 5. `n_cp_loo` emitted explicitly |
| 2 | **Gate misspecified.** `log(bw)` entered linearly, but the true relationship is non-monotone (tight is good, *zero* is terrible), so a linear logit cannot represent it. 0.7 precision at 2.2% coverage | bandwidth entered as **binned deciles + an explicit no-estimate class**, and benchmarked against a one-line band rule |
| 3 | **Placeholder co-location.** `bw = 0.000` *and* `err = 0.000` cases are shared-coordinate clusters (Block B: 909,820 parties in 16,824 clusters; one Pittsburgh point holds 29,780). They manufacture perfect scores | clusters ≥ `COLOC_MIN_CLUSTER` detected and excluded from **both** truth and clouds; population cost reported |
| 4 | **`is_hub` was wrong.** V0-minus-P99_9 captures hubs *plus every node whose only path ran through one* — 1,206,594 nodes, of which 2,961 were real | hubs defined by the **median-multiple degree rule** (robust to hub count, unlike a percentile); the ladder set kept as a separate diagnostic |
| 5 | **k-curve confounded.** `n` fell 1.9M → 10k across k, so each k was a different population and the rise after k=5 was composition, not evidence hurting | **fixed cohort**: targets with ≥ `K_MAX` located neighbours, so the population is constant across k |

Promoted to a standing rule: **accuracy is always reported conditional on
`geo_registered_vs_flow_km`.** The v1 pooled figure mixed estimator error with
ground-truth error at roughly a 6× ratio — 293 km pooled versus 46 km where
the registered pin is actually representative.

---

## What this notebook answers

Counterparty nodes have no address. The study runs on **customers**, whose
address is known: hide it, predict it back from neighbours, measure the error.

At degree 1 the error is bimodal by construction — a neighbourhood retailer
pins a target within kilometres; a card processor returns the population
prior. **The deliverable is therefore a gate, not a centroid.** Three outputs:

1. **Error-vs-evidence curve** — error as a function of located neighbours
2. **Locatability gate** — calibrated P(target within R km)
3. **Neighbour informativeness ranking** — which sectors and hubs carry location

## Method in one box

Each neighbour *j* is a kernel over where *its* counterparties live.

- **Centre** = *j*'s counterparty centroid (the target *is* one of them), not
  its registered pin.
- **Width** = *j*'s angular dispersion, so a processor self-mutes.

Using mean resultant length makes the leave-one-out exact:

```
S = Σw    V = Σw·u    R̄ = |V|/S
S' = S − w_i          V' = V − w_i·u_i          R̄' = |V'|/S'
centre' = V'/|V'|     width' = R_EARTH·sqrt(2(1−R̄'))     [NaN if S' < 2]
```

`R̄` is Block F's `geo_R`, so the bandwidth is consistent with the published
metric rather than a parallel invention.

---
## 0. Setup

In [ ]:
import os, re, math, time, json, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
import plotly.io as pio
pio.renderers.default = "notebook"; pio.templates.default = "plotly_white"
PALETTE = px.colors.qualitative.Safe

# ================================ CONFIG ==================================
WINDOW  = "last3"       # "last3" | "all"   <- staleness vs volume experiment
VERSION = "V0"          # "V0"    | "P99_9" <- hub-touching edges in / out

METRIC_TABLE  = "bdahd01p_dlcdi1_cdi_tm.cust_c2c_metrics"
METRIC_SOURCE = "table"                 # "table" | "parquet"
METRIC_GLOB   = "/user/pk36814/metrics/node/*.parquet"
EDGE_GLOB     = "../data/cust_*.csv"    # snapshots: source,dest,amount,volume
EDGE_FORMAT   = "csv"                   # "csv" | "parquet"

OUT_ROOT = "../metrics/locatability"
OUT = os.path.join(OUT_ROOT, f"{VERSION}_{WINDOW}")
os.makedirs(OUT, exist_ok=True)

# ---- guards (FIX 1) -------------------------------------------------------
# A neighbour left with ONE counterparty after the LOO has R̄'=1 and a
# bandwidth of exactly 0 — maximally confident, and in the v1 run 300 km
# wrong across 1.17M pairs. Zero dispersion is not "tight", it is "no
# estimate". Mirrors Block F's own MIN_CP_FOR_SPREAD / _FOR_ENTROPY.
MIN_CP_BW       = 2      # below this, bandwidth is NaN
MIN_CP_BW_TRUST = 5      # below this, bandwidth exists but is not trusted
# NOTE: bw_ok means the dispersion estimate is RELIABLE, not that the
# dispersion is SMALL. A neighbour with >=5 counterparties skews larger and
# more national, so bw_ok is not a quality filter — the first run showed it
# gives no accuracy lift at all. The quality filter is TIGHT_BW_KM below.
TIGHT_BW_KM = None       # set empirically in §6 from the decile-0 edge

# ---- co-location (FIX 3) --------------------------------------------------
# Placeholder geocoding puts tens of thousands of parties on one coordinate.
# Both target truth and neighbour clouds are corrupted by it.
EXCLUDE_COLOCATED = True
# Lowered from 25 after the first v2 run: at 25 only 9,080 parties were
# caught against Block B's ~910k in clusters, and sh_bw_zero stayed at 0.3%
# (~46k pairs) — exact-duplicate coordinates surviving in clusters below the
# threshold. A cluster of 5 on one coordinate is already a geocoding artefact.
COLOC_MIN_CLUSTER = 5
COLOC_ROUND       = 5    # decimal places at which coordinates are "identical"

# ---- hubs (FIX 4) ---------------------------------------------------------
# Median-multiple, not a percentile: a percentile flags a fixed fraction by
# construction and breaks when several hubs cluster at similar degree.
HUB_DEGREE_MULT = 15

# analysis parameters
K_MAX          = 20      # largest neighbour count in the curve, and the
                         # fixed-cohort threshold (FIX 5)
HARNESS_REPS   = 5
HIT_RADII_KM   = [25, 50, 250]
REPRESENTATIVE_GAP_KM = 25
MIN_R_BAR = 1e-9
SEED = 17

print(f"VERSION={VERSION}  WINDOW={WINDOW}\nout -> {OUT}")

In [ ]:
from pyspark.sql import SparkSession, functions as F, Window as W

spark = (SparkSession.builder.appName(f"pkg_locatability_{VERSION}_{WINDOW}")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.sql.shuffle.partitions", "600")
         .config("spark.sql.autoBroadcastJoinThreshold", str(64 * 1024 * 1024))
         .enableHiveSupport().getOrCreate())
spark.sparkContext.setLogLevel("WARN")
R_EARTH_KM = 6371.0088

def to_pd(sdf, label="", max_rows=3_000_000):
    t = time.time(); n = sdf.count()
    if n > max_rows:
        raise MemoryError(f"[{label}] {n:,} rows > max_rows={max_rows:,}; "
                          f"aggregate in Spark first.")
    df = sdf.toPandas()
    print(f"[{label}] {n:,} rows | {time.time()-t:,.1f}s")
    return df

def unit_cols(df, lat="lat", lon="lon", pre=""):
    la, lo = F.radians(F.col(lat)), F.radians(F.col(lon))
    return (df.withColumn(pre+"ux", F.cos(la) * F.cos(lo))
              .withColumn(pre+"uy", F.cos(la) * F.sin(lo))
              .withColumn(pre+"uz", F.sin(la)))

def gc_km(lat1, lon1, lat2, lon2):
    p1, p2 = F.radians(lat1), F.radians(lat2)
    dp, dl = p2 - p1, F.radians(lon2) - F.radians(lon1)
    a = (F.sin(dp/2)**2 + F.cos(p1)*F.cos(p2)*F.sin(dl/2)**2)
    return F.lit(2*R_EARTH_KM) * F.asin(F.sqrt(F.least(a, F.lit(1.0))))

def vec_to_latlon(df, vx, vy, vz, out_lat, out_lon):
    "Resultant vector -> lat/lon on the sphere."
    n = F.sqrt(F.col(vx)**2 + F.col(vy)**2 + F.col(vz)**2)
    return (df.withColumn("_n", n)
              .withColumn(out_lat, F.degrees(F.asin(F.col(vz)/F.col("_n"))))
              .withColumn(out_lon, F.degrees(F.atan2(F.col(vy), F.col(vx))))
              .drop("_n"))

def qbucket(df, col, n_bins=10, rel_err=0.01):
    """Decile bucketing via approxQuantile, NOT ntile.

    `ntile` over an unpartitioned window sorts the whole frame into one
    partition — on a pair table of millions of rows that is a driver-side
    collapse, not a slowdown. NULLs get their own bucket (-1) rather than
    being silently folded into bucket 0, which matters now that an unguarded
    bandwidth is NULL rather than 0.
    """
    qs = sorted(set(df.approxQuantile(col, [i/n_bins for i in range(1, n_bins)],
                                      rel_err)))
    e = F.when(F.col(col).isNull(), F.lit(-1))
    for i, v in enumerate(qs):
        e = e.when(F.col(col) <= F.lit(v), F.lit(i))
    return df.withColumn("bin", e.otherwise(F.lit(len(qs)))), qs

---
## 1. Metrics dimension, hub set, and the co-location registry

In [ ]:
SDF_M = (spark.table(METRIC_TABLE) if METRIC_SOURCE == "table"
         else spark.read.parquet(METRIC_GLOB))
COLS = [c.lower() for c in SDF_M.columns]
SDF_M = SDF_M.toDF(*COLS)

def have(*c):
    for x in c:
        if x in COLS: return x
    return None

NODE, TIME, VER = have("mdm_id", "node"), have("time_key"), have("version")
assert NODE and TIME and VER, f"missing keys: {NODE}, {TIME}, {VER}"
print(f"join key = {NODE}  |  dtype = {dict(SDF_M.dtypes)[NODE]}")

MONTHS = sorted(r[0] for r in SDF_M.select(TIME).distinct().collect())
WIN_MONTHS = MONTHS[-3:] if WINDOW == "last3" else MONTHS
print(f"{len(MONTHS)} months available; window '{WINDOW}' uses "
      f"{len(WIN_MONTHS)}: {WIN_MONTHS[0]} .. {WIN_MONTHS[-1]}")

attr_cols = [c for c in [NODE, "lat", "lon", "geo_status", "state", "zip3",
                         "naics2", "naics_desc", "cust_name", "node_type",
                         "entity_type", "party_type", "in_degree", "out_degree",
                         "in_strength", "out_strength", "geo_spread_km",
                         "geo_reach_p50_km", "geo_registered_vs_flow_km",
                         "geo_r"] if c in COLS]
ATTR = (SDF_M.filter((F.col(VER) == "V0") & (F.col(TIME) == WIN_MONTHS[-1]))
             .select(*attr_cols).withColumn("node", F.col(NODE).cast("string")))
if NODE != "node": ATTR = ATTR.drop(NODE)
ATTR = (ATTR.withColumn("deg_tot",
            F.coalesce(F.col("in_degree"), F.lit(0)) +
            F.coalesce(F.col("out_degree"), F.lit(0)))
            .withColumn("has_geo", (F.col("geo_status") == "valid").cast("int")))

In [ ]:
# ---- FIX 4: hubs by median-multiple degree, not by ladder exclusion --------
# v1 used "present in V0, absent from P99_9". That set was 1,206,594 nodes,
# because it also captures every node whose ONLY connection ran through a
# hub. Only ~3k were real hubs, so `is_hub` was largely tagging isolated
# small nodes — visible in the v1 gate as a large negative coefficient.
med_deg = ATTR.approxQuantile("deg_tot", [0.5], 0.01)[0]
HUB_CUT = max(HUB_DEGREE_MULT * med_deg, 2.0)
ATTR = ATTR.withColumn("is_hub", (F.col("deg_tot") >= F.lit(HUB_CUT)).cast("int"))
n_hub = ATTR.filter("is_hub = 1").count()
print(f"median degree {med_deg:,.0f} -> hub cut {HUB_CUT:,.0f} "
      f"({HUB_DEGREE_MULT}x median) -> {n_hub:,} hubs")
if n_hub == 0:
    print("WARNING: no hubs at this cut. In a graph where most nodes are "
          "degree 1-2 the median-multiple rule is well behaved, but on a "
          "dense subgraph it can exceed max degree. Check the degree "
          "distribution before lowering HUB_DEGREE_MULT — a percentile is "
          "NOT the fallback, it flags a fixed fraction by construction.")
    print(ATTR.approxQuantile("deg_tot", [.5,.9,.99,.999,1.0], 0.01))

# the ladder-derived set is retained as a DIAGNOSTIC only
v0  = (SDF_M.filter((F.col(VER) == "V0") & F.col(TIME).isin(WIN_MONTHS))
            .select(F.col(NODE).cast("string").alias("node")).distinct())
p99 = (SDF_M.filter((F.col(VER) == "P99_9") & F.col(TIME).isin(WIN_MONTHS))
            .select(F.col(NODE).cast("string").alias("node")).distinct())
LADDER_OUT = v0.join(p99, "node", "left_anti").cache()
n_ladder = LADDER_OUT.count()
ATTR = (ATTR.join(LADDER_OUT.withColumn("lx", F.lit(1)), "node", "left")
            .withColumn("ladder_excluded", F.coalesce(F.col("lx"), F.lit(0)))
            .drop("lx")).cache()
print(f"ladder-excluded (V0 minus P99_9): {n_ladder:,} nodes — "
      f"{n_ladder/max(v0.count(),1):.1%} of V0. This is NOT the hub set; it is "
      f"hubs plus everything that only reached the graph through one.")
print(f"overlap with the degree-based hub set: "
      f"{ATTR.filter('is_hub = 1 AND ladder_excluded = 1').count():,}")

In [ ]:
# ---- FIX 3: the co-location registry --------------------------------------
# Placeholder geocoding stacks parties on a single coordinate. Such a point
# yields R̄=1 (bandwidth 0) for any neighbour whose cloud sits there, and
# error 0 whenever target and neighbour share it — manufacturing perfect
# scores from a data-quality artefact.
GEO = ATTR.filter("has_geo = 1").select("node", "lat", "lon")
GEO = (GEO.withColumn("plat", F.round(F.col("lat"), COLOC_ROUND))
          .withColumn("plon", F.round(F.col("lon"), COLOC_ROUND)))
CLUST = (GEO.groupBy("plat", "plon").agg(F.count("*").alias("n_at_point"))
            .filter(F.col("n_at_point") >= COLOC_MIN_CLUSTER).cache())
n_clust = CLUST.count()
COLOC = (GEO.join(CLUST, ["plat", "plon"])
            .select("node", "n_at_point",
                    F.concat_ws("_", "plat", "plon").alias("coloc_id")).cache())
n_coloc = COLOC.count(); n_geo = GEO.count()
print(f"co-located clusters (>= {COLOC_MIN_CLUSTER} parties on one point): "
      f"{n_clust:,}")
print(f"parties inside them: {n_coloc:,} of {n_geo:,} located "
      f"({n_coloc/max(n_geo,1):.1%})")
print("\ntop shared coordinates:")
CLUST.orderBy(F.desc("n_at_point")).show(8, truncate=False)

ATTR = (ATTR.join(COLOC.select("node", F.lit(1).alias("cl")), "node", "left")
            .withColumn("is_coloc", F.coalesce(F.col("cl"), F.lit(0)))
            .drop("cl")).cache()

In [ ]:
# LOC is the set usable as truth AND as cloud members.
LOC_ALL = ATTR.filter("has_geo = 1").select("node", "lat", "lon", "is_coloc")
if EXCLUDE_COLOCATED:
    LOC = LOC_ALL.filter("is_coloc = 0").drop("is_coloc")
    print(f"EXCLUDE_COLOCATED=True -> {LOC.count():,} usable located nodes "
          f"({n_coloc:,} dropped)")
    print("Excluded from BOTH roles deliberately: as a target its truth is a "
          "placeholder centroid, and as a cloud member it collapses the "
          "neighbour's dispersion toward zero.")
else:
    LOC = LOC_ALL.drop("is_coloc")
    print(f"EXCLUDE_COLOCATED=False -> {LOC.count():,} located nodes, "
          f"artefacts INCLUDED. Results are not comparable to a guarded run.")
LOC = unit_cols(LOC).select("node", "lat", "lon", "ux", "uy", "uz").cache()

---
## 2. Edges, pooled over the window

In [ ]:
if EDGE_FORMAT == "csv":
    E = (spark.read.option("header", True).csv(EDGE_GLOB)
              .withColumn("amount", F.col("amount").cast("double")))
else:
    E = spark.read.parquet(EDGE_GLOB)
E = E.toDF(*[c.lower() for c in E.columns])
if "time_key" not in E.columns:
    E = E.withColumn("time_key",
                     F.regexp_extract(F.input_file_name(), r"(\d{4}-\d{2})", 1))
E = (E.withColumn("source", F.col("source").cast("string"))
       .withColumn("dest",   F.col("dest").cast("string"))
       .filter(F.col("time_key").isin(WIN_MONTHS))
       .filter(F.col("source") != F.col("dest")))
print(f"edge rows in window: {E.count():,}")

# JOIN INTEGRITY. The 2026-07 incident was an int64/str mismatch that typed
# every counterparty 'unknown' while producing plausible output.
probe = E.select("source").limit(200_000).distinct()
mrate = (probe.join(ATTR.select("node"), probe.source == F.col("node"), "left")
              .agg(F.avg(F.col("node").isNotNull().cast("double"))).first()[0])
print(f"edge->metric join match rate: {mrate:.2%}")
assert mrate > 0.5, ("join match rate below 50% — check that source/dest and "
                     f"{NODE} are both strings and share an id space")

In [ ]:
fwd = E.select(F.col("source").alias("a"), F.col("dest").alias("b"),
               F.col("amount").alias("amt_ab"), F.lit(0.0).alias("amt_ba"),
               "time_key")
rev = E.select(F.col("dest").alias("a"), F.col("source").alias("b"),
               F.lit(0.0).alias("amt_ab"), F.col("amount").alias("amt_ba"),
               "time_key")
PAIR = (fwd.union(rev).groupBy("a", "b")
          .agg(F.sum("amt_ab").alias("amt_out"),
               F.sum("amt_ba").alias("amt_in"),
               F.countDistinct("time_key").alias("n_months"))
          .withColumn("amt", F.col("amt_out") + F.col("amt_in")))

if VERSION == "P99_9":
    HUBIDS = ATTR.filter("is_hub = 1").select("node")
    PAIR = (PAIR.join(HUBIDS.withColumn("hb", F.lit(1)),
                      PAIR.b == HUBIDS.node, "left").filter("hb is null")
                .drop("node", "hb")
                .join(HUBIDS.withColumn("ha", F.lit(1)),
                      F.col("a") == F.col("node"), "left").filter("ha is null")
                .drop("node", "ha"))
PAIR = PAIR.cache()
print(f"directed-pair rows (target a -> neighbour b): {PAIR.count():,}")

## 3. Neighbour clouds and the guarded leave-one-out

`NB` totals each neighbour's located counterparties; `loo_block` subtracts the
target's own contribution so the centre and width were computed **without
seeing the answer**.

**The guard is the headline change.** `n_cp_loo` is the number of counterparties
left after the subtraction. Below 2 there is no dispersion to measure and the
bandwidth is NaN — not 0. In v1 that case was 1.17M pairs presenting as
maximally confident while being ~300 km wrong, and it sat at the *good* end of
the feature, which is why a linear gate could not use bandwidth properly.

In [ ]:
CP = (PAIR.join(LOC.withColumnRenamed("node", "a")
                   .withColumnRenamed("lat", "a_lat")
                   .withColumnRenamed("lon", "a_lon"), "a"))

NB = (CP.groupBy("b").agg(
        F.count("*").alias("n_cp_loc"),
        F.sum("amt").alias("amt_loc"),
        F.sum("ux").alias("cV_x"), F.sum("uy").alias("cV_y"),
        F.sum("uz").alias("cV_z"),
        F.sum(F.col("ux")*F.col("amt")).alias("aV_x"),
        F.sum(F.col("uy")*F.col("amt")).alias("aV_y"),
        F.sum(F.col("uz")*F.col("amt")).alias("aV_z"))).cache()
print(f"nodes usable as evidence (>=1 located counterparty): {NB.count():,}")

In [ ]:
def loo_block(df, pre, S, Vx, Vy, Vz, w, n_left):
    """Exact leave-one-out for centre and width, with the degeneracy guard.

    Removing the target is a subtraction, not a recomputation, so the LOO is
    exact at one subtraction per pair. Without it the neighbour's cloud
    statistic contains the node being predicted, which is circular at low
    neighbour degree.

    GUARD (FIX 1): with fewer than MIN_CP_BW counterparties remaining, R̄'=1
    identically and the width would be exactly 0 — a confident, wrong kernel.
    The width is NaN there instead, and `{pre}bw_ok` marks whether it is
    trustworthy at all.
    """
    d = (df.withColumn(f"{pre}S",  F.col(S)  - F.col(w))
           .withColumn(f"{pre}Vx", F.col(Vx) - F.col(w)*F.col("ux"))
           .withColumn(f"{pre}Vy", F.col(Vy) - F.col(w)*F.col("uy"))
           .withColumn(f"{pre}Vz", F.col(Vz) - F.col(w)*F.col("uz")))
    d = d.withColumn(f"{pre}Vn", F.sqrt(F.col(f"{pre}Vx")**2 +
                                        F.col(f"{pre}Vy")**2 +
                                        F.col(f"{pre}Vz")**2))
    d = (d.withColumn(f"{pre}rbar", F.when(F.col(f"{pre}S") > 0,
                        F.least(F.lit(1.0), F.col(f"{pre}Vn")/F.col(f"{pre}S"))))
           .withColumn(f"{pre}lat", F.when(F.col(f"{pre}Vn") > MIN_R_BAR,
                        F.degrees(F.asin(F.col(f"{pre}Vz")/F.col(f"{pre}Vn")))))
           .withColumn(f"{pre}lon", F.when(F.col(f"{pre}Vn") > MIN_R_BAR,
                        F.degrees(F.atan2(F.col(f"{pre}Vy"), F.col(f"{pre}Vx"))))))
    raw = F.lit(R_EARTH_KM) * F.sqrt(F.greatest(
            F.lit(0.0), F.lit(2.0)*(F.lit(1.0) - F.col(f"{pre}rbar"))))
    return (d.withColumn(f"{pre}bw_km",
                         F.when(F.col(n_left) >= MIN_CP_BW, raw))
             .withColumn(f"{pre}bw_ok",
                         (F.col(n_left) >= MIN_CP_BW_TRUST).cast("int")))

L = (CP.join(NB, "b").withColumn("w1", F.lit(1.0))
       .withColumn("n_cp_loo", F.col("n_cp_loc") - F.lit(1))
       .transform(lambda d: loo_block(d, "kc_", "n_cp_loc", "cV_x", "cV_y",
                                      "cV_z", "w1", "n_cp_loo"))
       .transform(lambda d: loo_block(d, "ka_", "amt_loc", "aV_x", "aV_y",
                                      "aV_z", "amt", "n_cp_loo")))
L = (L.withColumn("err_cnt_km", gc_km(F.col("a_lat"), F.col("a_lon"),
                                      F.col("kc_lat"), F.col("kc_lon")))
       .withColumn("err_amt_km", gc_km(F.col("a_lat"), F.col("a_lon"),
                                       F.col("ka_lat"), F.col("ka_lon")))
       .filter(F.col("kc_S") > 0))

NBATTR = ATTR.select(F.col("node").alias("b"),
                     F.col("naics2").alias("b_naics2"),
                     F.col("node_type").alias("b_node_type"),
                     F.col("state").alias("b_state"),
                     F.col("cust_name").alias("b_name"),
                     F.col("is_hub").alias("b_is_hub"),
                     F.col("deg_tot").alias("b_deg"))
L = L.join(NBATTR, "b", "left").cache()
n_pairs = L.count()
print(f"LOO pair rows (one degree-1 experiment each): {n_pairs:,}")

In [ ]:
# Guard audit: how much of the census the fix removes from the confident end.
aud = (L.agg(F.count("*").alias("pairs"),
             F.avg((F.col("n_cp_loo") < MIN_CP_BW).cast("double")).alias("sh_no_bw"),
             F.avg((F.col("kc_bw_ok") == 0).cast("double")).alias("sh_untrusted"),
             F.avg(F.col("kc_bw_km").isNull().cast("double")).alias("sh_bw_null"),
             F.avg(((F.col("kc_bw_km") == 0)).cast("double")).alias("sh_bw_zero"))
        .first().asDict())
print(json.dumps({k: (float(v) if v is not None else None)
                  for k, v in aud.items()}, indent=2))
print(f"\nsh_bw_zero must be ~0: an exact zero bandwidth is the v1 defect.")
print(f"sh_no_bw is the share with no dispersion estimate at all — in v1 these "
      f"were scored as the MOST confident pairs in the study.")
L.select("a","b","n_cp_loo","kc_bw_km","kc_bw_ok","err_cnt_km",
         "b_naics2","b_is_hub").show(5)

---
## 4. Baselines

The **footprint prior** is the one that matters: PNC customers concentrate in
a handful of states, so a model that learned only that will look skilful. In
the v1 run the prior scored `hit@50 = 0.014%`, which makes it a genuinely
weak opponent at tight radii — and therefore a fair one to quote against.

In [ ]:
prior = (LOC.agg(F.avg("ux").alias("x"), F.avg("uy").alias("y"),
                 F.avg("uz").alias("z")).first())
pn = math.sqrt(prior.x**2 + prior.y**2 + prior.z**2)
PRIOR_LAT = math.degrees(math.asin(prior.z/pn))
PRIOR_LON = math.degrees(math.atan2(prior.y, prior.x))
print(f"PNC footprint prior centroid: {PRIOR_LAT:.3f}, {PRIOR_LON:.3f}")

base = (LOC.withColumn("err_km", gc_km(F.col("lat"), F.col("lon"),
                                       F.lit(PRIOR_LAT), F.lit(PRIOR_LON)))
           .agg(F.expr("percentile_approx(err_km, 0.5)").alias("median_km"),
                F.expr("percentile_approx(err_km, 0.9)").alias("p90_km"),
                *[F.avg((F.col("err_km") <= r).cast("double")).alias(f"hit_{r}")
                  for r in HIT_RADII_KM]).first().asDict())
BASELINE = {"estimator": "footprint_prior", **base}
print(json.dumps(BASELINE, indent=2, default=float))

---
## 5. Degree 1 — a census, reported conditional on ground-truth quality

Every located pair **is** a degree-1 experiment, so this is a census, not a
sample.

**Standing rule, promoted after v1:** never quote a single pooled accuracy
number. Error against the registered pin mixes estimator error with
ground-truth error, and in v1 that ratio was roughly 6× — 293 km pooled
against 46 km where the pin is representative. The pooled figure is the one
that would have been briefed, and it would have understated the capability by
an order of magnitude.

In [ ]:
GAP = ATTR.select(F.col("node").alias("a"),
                  F.col("geo_registered_vs_flow_km").alias("a_gap"),
                  F.col("geo_spread_km").alias("a_spread"),
                  F.col("state").alias("a_state"),
                  F.col("entity_type").alias("a_etype"))
LG = L.join(GAP, "a", "left").cache()

TRUTH_SETS = [
    ("all targets",                    F.lit(True)),
    ("pin representative (<=25km gap)", F.col("a_gap") <= REPRESENTATIVE_GAP_KM),
    ("pin unrepresentative",            F.col("a_gap") >  REPRESENTATIVE_GAP_KM),
    ("target is local (<50km spread)",  F.col("a_spread") < 50),
    # bw_ok is RELIABILITY, not tightness — it gave zero lift in the first
    # run and is kept only to make that visible.
    ("bw estimate reliable (n_cp_loo>=5)", F.col("kc_bw_ok") == 1),
    ("representative AND bw reliable",
     (F.col("a_gap") <= REPRESENTATIVE_GAP_KM) & (F.col("kc_bw_ok") == 1)),
]
rows = []
for name, cond in TRUTH_SETS:
    r = (LG.filter(cond).agg(
            F.count("*").alias("n"),
            F.expr("percentile_approx(err_cnt_km,0.5)").alias("median_km"),
            F.expr("percentile_approx(err_cnt_km,0.9)").alias("p90_km"),
            F.expr("percentile_approx(err_amt_km,0.5)").alias("median_amt_km"),
            *[F.avg((F.col("err_cnt_km") <= x).cast("double")).alias(f"hit_{x}")
              for x in HIT_RADII_KM]).first().asDict())
    rows.append({"truth_set": name, **r})
TRUTH = pd.DataFrame(rows)
print(TRUTH.to_string(index=False))
TRUTH.to_csv(os.path.join(OUT, "truth_sensitivity.csv"), index=False)
d1 = TRUTH[TRUTH.truth_set == "all targets"].iloc[0].to_dict()
print(f"\nfootprint prior: median {BASELINE['median_km']:,.0f} km, "
      f"hit@50 {BASELINE['hit_50']:.4%}")
print("median_amt_km is the amount-weighted kernel. If it is worse than the "
      "count-weighted median, dollars do not carry location — the largest "
      "edge is disproportionately a processor.")

In [ ]:
h = (L.filter(F.col("err_cnt_km").isNotNull())
       .select(F.when(F.col("err_cnt_km") < 1, 0.0)
                .otherwise(F.log10(F.col("err_cnt_km"))).alias("le"),
               F.col("kc_bw_ok"))
       .withColumn("bin", F.floor(F.col("le")*4)/4)
       .groupBy("bin", "kc_bw_ok").count().orderBy("bin"))
hp = to_pd(h, "error histogram")
hp["evidence"] = np.where(hp.kc_bw_ok == 1, "bandwidth trusted",
                          "no / weak dispersion estimate")
fig = px.bar(hp, x="bin", y="count", color="evidence", barmode="stack",
             title="Degree-1 error distribution (log10 km), split by whether "
                   "the neighbour had a usable dispersion estimate",
             color_discrete_sequence=PALETTE)
for r in HIT_RADII_KM:
    fig.add_vline(x=math.log10(r), line_dash="dash", line_color="crimson",
                  annotation_text=f"{r} km")
fig.update_layout(height=440, xaxis_title="log10 error km", yaxis_title="pairs")
fig.show()
print("The two colours should separate. If the untrusted group is spread "
      "across the whole range while the trusted group concentrates left, the "
      "gate has something to work with and v1's single accuracy number was "
      "averaging two different populations.")

---
## 6. What makes a neighbour informative

The claim: **kernel width predicts error, and everything else acts through
it.** If that holds the gate needs one feature and is trivially explainable.

The v1 run appeared to show a non-monotone relationship. It was an artefact of
the unguarded bandwidth: deciles 0–1 were the `bw = 0` degeneracy. With the
guard those pairs are a separate class and the relationship should be
monotone from the first decile.

In [ ]:
def by_bin(col, label, n_bins=10, categorical=False, min_n=500, frame=None):
    d = (frame if frame is not None else L).filter(F.col("err_cnt_km").isNotNull())
    if categorical:
        d = d.withColumn("bin", F.col(col).cast("string"))
    else:
        d, _ = qbucket(d, col, n_bins)
    g = (d.groupBy("bin").agg(
            F.count("*").alias("n"),
            (F.avg(col) if not categorical
             else F.lit(None).cast("double")).alias("bin_value"),
            F.expr("percentile_approx(err_cnt_km, 0.5)").alias("median_err_km"),
            F.expr("percentile_approx(err_cnt_km, 0.9)").alias("p90_err_km"),
            *[F.avg((F.col("err_cnt_km") <= r).cast("double")).alias(f"hit_{r}")
              for r in HIT_RADII_KM])
          .filter(F.col("n") >= min_n).orderBy("bin"))
    out = to_pd(g, label); out["feature"] = label
    return out

BW = by_bin("kc_bw_km", "kernel bandwidth decile")
print(BW.to_string(index=False))
print("\nbin = -1 is the NO-ESTIMATE class (n_cp_loo < 2). It is reported "
      "separately, never as decile 0 — that conflation is what broke v1.")
BWv = BW[BW.bin.astype(str) != "-1"]
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_bar(x=BW.bin.astype(str), y=BW.n, name="pairs", marker_color="#dfe6e9")
fig.add_scatter(x=BWv.bin.astype(str), y=BWv.median_err_km, name="median err km",
                mode="lines+markers", line=dict(width=3), secondary_y=True)
fig.add_scatter(x=BWv.bin.astype(str), y=BWv[f"hit_50"]*1000,
                name="hit@50km (x1000)", mode="lines+markers",
                line=dict(dash="dot"), secondary_y=True)
fig.update_layout(height=440, title="Neighbour kernel bandwidth vs prediction "
                  "error — should now be monotone across the valid deciles",
                  xaxis_title="bandwidth decile (-1 = no estimate)")
fig.show()

In [ ]:
# The no-estimate class (bin -1) is bimodal: median 714 km but hit@50 22%,
# and the gate gave it a large positive coefficient. It is averaging two
# populations, so split it before letting the gate use it as one feature.
NOEST = L.filter(F.col("kc_bw_km").isNull() & F.col("err_cnt_km").isNotNull())
print(f"no-estimate pairs: {NOEST.count():,}")
NE = by_bin("b_deg", "NO-ESTIMATE by neighbour degree", frame=NOEST, min_n=200)
print(NE.to_string(index=False))
NEH = by_bin("b_is_hub", "NO-ESTIMATE by hub flag", categorical=True,
             frame=NOEST, min_n=200)
print(NEH.to_string(index=False))
print("\nA neighbour with ONE located counterparty is not one thing: it can be "
      "a genuinely tiny local node (informative) or a large node almost all of "
      "whose counterparties are unlocated (uninformative). Degree separates "
      "them, which is why the gate carries n_cp and degree alongside the "
      "no-estimate indicator rather than a single flag.")

DEG = by_bin("b_deg", "neighbour degree decile")
AMT = by_bin("amt", "edge amount decile")
MON = by_bin("n_months", "months active", categorical=True)
HUB = by_bin("b_is_hub", "hub flag", categorical=True)
NAI = by_bin("b_naics2", "neighbour NAICS2", categorical=True, min_n=2000)

fig = make_subplots(rows=2, cols=2, subplot_titles=(
    "neighbour degree decile", "edge amount decile",
    "months the relationship was active", "hub flag (degree-based)"))
for i, (d, r, c) in enumerate([(DEG,1,1), (AMT,1,2), (MON,2,1), (HUB,2,2)]):
    fig.add_scatter(x=d.bin.astype(str), y=d.median_err_km, mode="lines+markers",
                    row=r, col=c, showlegend=False,
                    line=dict(color=PALETTE[i], width=3))
fig.update_layout(height=620, title="Median degree-1 error by neighbour "
                                    "attribute (lower is more informative)")
fig.show()

NAI = NAI.sort_values("median_err_km")
fig = px.bar(NAI, x="bin", y="median_err_km", hover_data=["n", "hit_50"],
             title="Which sectors locate? Median degree-1 error by neighbour "
                   "NAICS2 (ordered)", color_discrete_sequence=PALETTE)
fig.update_layout(height=430, xaxis_title="neighbour naics2",
                  yaxis_title="median error km")
fig.show()
print(NAI[["bin","n","median_err_km","p90_err_km","hit_50"]].to_string(index=False))
print("\nNAICS-null is mostly persons and was the single largest group in v1 "
      "(42% of pairs). Read it as a population, not a data gap.")

In [ ]:
# Does sector act ONLY through bandwidth? Restricted to pairs where the
# bandwidth is real, otherwise the conditioning variable is partly missing.
LV = L.filter((F.col("err_cnt_km").isNotNull()) & (F.col("kc_bw_km").isNotNull()))
d, _ = qbucket(LV, "kc_bw_km", 10)
d = d.withColumnRenamed("bin", "bwd")
raw = (d.groupBy("b_naics2").agg(F.expr("percentile_approx(err_cnt_km,0.5)")
        .alias("raw_median"), F.count("*").alias("n")).filter("n >= 2000"))
cond = (d.groupBy("b_naics2", "bwd")
          .agg(F.expr("percentile_approx(err_cnt_km,0.5)").alias("m"),
               F.count("*").alias("n")).filter("n >= 200")
          .groupBy("b_naics2").agg(
              (F.sum(F.col("m")*F.col("n"))/F.sum("n")).alias("cond_median")))
S = to_pd(raw.join(cond, "b_naics2"), "sector conditional")
shrink = 1 - (S.cond_median.std() / max(S.raw_median.std(), 1e-9))
fig = px.scatter(S, x="raw_median", y="cond_median", text="b_naics2", size="n",
                 title="Sector effect before vs after conditioning on kernel "
                       "bandwidth (bandwidth-valid pairs only)",
                 color_discrete_sequence=PALETTE)
mx = float(max(S.raw_median.max(), S.cond_median.max()))
fig.add_shape(type="line", x0=0, y0=0, x1=mx, y1=mx, line=dict(dash="dash"))
fig.update_traces(textposition="top center")
fig.update_layout(height=520, xaxis_title="raw median err km",
                  yaxis_title="bandwidth-conditioned median err km")
fig.show()
print(f"spread of the sector effect shrinks by {shrink:.1%} once bandwidth is "
      f"held fixed.")
print("Near 100%: sector is a proxy for footprint width and belongs in the "
      "PRIOR on bandwidth. Well below: sector earns its own term.")

---
## 7. The locatability gate

Output a **calibrated probability that the target is within R km**, not a
point estimate.

Two changes from v1. Bandwidth enters as **binned deciles plus an explicit
no-estimate class**, because the relationship is not monotone in the raw value
and a linear logit cannot represent "tight is good, zero is meaningless".
And the fitted gate is benchmarked against a **one-line band rule** — in v1 a
single bandwidth decile beat the whole model by 4× on coverage, which is the
kind of thing worth knowing before shipping a model.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.calibration import calibration_curve
from sklearn.metrics import roc_auc_score, precision_recall_curve

GATE_R = 50
feat_sdf = (L.select(
        F.col("kc_bw_km").alias("bw"),
        F.col("kc_bw_ok").alias("bw_ok"),
        F.col("n_cp_loo").cast("double").alias("n_cp"),
        F.coalesce(F.col("b_deg"), F.lit(0)).cast("double").alias("deg"),
        F.col("amt"), F.col("n_months").cast("double").alias("months"),
        F.col("b_is_hub").cast("double").alias("hub"),
        F.coalesce(F.col("b_naics2"), F.lit("NA")).alias("naics2"),
        (F.col("err_cnt_km") <= GATE_R).cast("int").alias("y"),
        F.col("err_cnt_km"))
     .filter(F.col("err_cnt_km").isNotNull()))
frac = min(1.0, 3_000_000 / max(feat_sdf.count(), 1))
G = to_pd(feat_sdf.sample(False, frac, seed=SEED), "gate features")
print(f"gate sample: {len(G):,} | base rate within {GATE_R} km: {G.y.mean():.2%}")
print(f"no-bandwidth share: {G.bw.isna().mean():.2%}")

tr, te = train_test_split(G, test_size=0.3, random_state=SEED, stratify=G.y)

# --- BASELINE 0: the one-line band rule -----------------------------------
# The first v2 run scanned ROUND numbers and shipped 1<bw<=10 at 0.697/4.0%,
# while bandwidth decile 0 gave 0.676 at 8.5% — twice the coverage. The grid
# was anchored on human-friendly cut points while the real boundary sits near
# the decile edge. Scan the EMPIRICAL quantiles instead.
qedge = [0.0] + list(tr.bw.dropna().quantile(
    [.02,.05,.08,.10,.15,.20,.25,.30,.40,.50]).round(2).unique())
print(f"candidate cut points from the bandwidth distribution: {qedge}")
cand = []
for i, lo in enumerate(qedge):
    for hi in qedge[i+1:]:
        m = tr.bw.between(lo, hi)
        if m.sum() < 5000: continue
        cand.append({"lo": lo, "hi": hi, "precision": tr.y[m].mean(),
                     "coverage": m.mean(),
                     "lift_x_cov": tr.y[m].mean() * m.mean()})
CANDS = pd.DataFrame(cand)
print("\ntop band rules by PRECISION (train):")
print(CANDS.sort_values("precision", ascending=False).head(6).to_string(index=False))
print("\ntop band rules by COVERAGE among those clearing the precision floor:")
# Choose MAX COVERAGE SUBJECT TO A PRECISION FLOOR, not max precision (which
# picks a 4%-coverage rule that is not a product) and not max
# precision x coverage (which drifts to wide bands whenever precision is
# locally flat). The floor is the product decision; state it explicitly.
BAND_MIN_PRECISION = 0.65
elig = CANDS[CANDS.precision >= BAND_MIN_PRECISION]
if len(elig) == 0:
    print(f"WARNING: no band reaches precision {BAND_MIN_PRECISION}; falling "
          f"back to the best available.")
    elig = CANDS
bb = elig.sort_values("coverage", ascending=False).iloc[0]
mte = te.bw.between(bb.lo, bb.hi)
BAND = {"rule": f"{bb.lo} < bw <= {bb.hi} km",
        "precision": float(te.y[mte].mean()), "coverage": float(mte.mean())}
print(f"\nBAND RULE on test: {BAND['rule']} -> precision {BAND['precision']:.3f}, "
      f"coverage {BAND['coverage']:.3%}")
TIGHT_BW_KM = float(bb.hi)
print(f"TIGHT_BW_KM set to {TIGHT_BW_KM} — this is the 'tight kernel' cut used "
      f"everywhere below, and the one the scoring function ships with.")

In [ ]:
# --- the fitted gate, with bandwidth BINNED (FIX 2) -----------------------
QS = tr.bw.dropna().quantile(np.linspace(0.1, 0.9, 9)).unique()
def bw_bin(s):
    b = np.digitize(s.fillna(-1).to_numpy(), np.r_[-0.5, QS]).astype(int)
    return pd.Series(np.where(s.isna().to_numpy(), -1, b), index=s.index)

enc = tr.groupby("naics2")["y"].agg(["mean", "size"]); gm = tr.y.mean()
enc["sm"] = (enc["mean"]*enc["size"] + gm*200) / (enc["size"] + 200)

def featurise(d):
    X = pd.DataFrame(index=d.index)
    X["log_ncp"]   = np.log1p(d.n_cp.clip(lower=0))
    X["log_deg"]   = np.log1p(d.deg.clip(lower=0))
    X["log_amt"]   = np.log1p(d.amt.clip(lower=0))
    X["months"]    = d.months.fillna(0)
    X["hub"]       = d.hub.fillna(0)
    X["naics_prior"] = d.naics2.map(enc["sm"]).fillna(gm)
    B = pd.get_dummies(bw_bin(d.bw).astype(int).rename("bwbin"), prefix="bw")
    return pd.concat([X, B], axis=1)

Xtr, Xte = featurise(tr), featurise(te)
Xte = Xte.reindex(columns=Xtr.columns, fill_value=0)
clf = LogisticRegression(max_iter=3000, C=1.0).fit(Xtr.fillna(0), tr.y)
te = te.copy(); te["p"] = clf.predict_proba(Xte.fillna(0))[:, 1]
auc = roc_auc_score(te.y, te.p)

# bandwidth-only, also binned — the auditable one-feature version
bcols = [c for c in Xtr.columns if c.startswith("bw_")]
clf1 = LogisticRegression(max_iter=3000).fit(Xtr[bcols], tr.y)
auc1 = roc_auc_score(te.y, clf1.predict_proba(Xte[bcols])[:, 1])
print(f"AUC full = {auc:.4f} | binned-bandwidth-only = {auc1:.4f}")
print(f"(v1, unbinned: 0.8102 / 0.7893)")
print("\ncoefficients:")
print(pd.Series(clf.coef_[0], index=Xtr.columns).sort_values().to_string())

In [ ]:
pt, pp = calibration_curve(te.y, te.p, n_bins=20, strategy="quantile")
fig = go.Figure()
fig.add_scatter(x=pp, y=pt, mode="lines+markers", name="model",
                line=dict(width=3, color=PALETTE[0]))
fig.add_shape(type="line", x0=0, y0=0, x1=1, y1=1, line=dict(dash="dash"))
fig.update_layout(height=460, title=f"Gate calibration — predicted vs actual "
                  f"P(within {GATE_R} km)", xaxis_title="predicted",
                  yaxis_title="observed")
fig.show()

prec, rec, thr = precision_recall_curve(te.y, te.p)
ops = []
for tp in (0.70, 0.80, 0.90, 0.95):
    ok = np.where(prec[:-1] >= tp)[0]
    if len(ok):
        i = ok[np.argmax(rec[ok])]
        ops.append({"target_precision": tp, "threshold": float(thr[i]),
                    "achieved_precision": float(prec[i]), "recall": float(rec[i]),
                    "share_of_pairs_flagged": float((te.p >= thr[i]).mean())})
OPS = pd.DataFrame(ops)
OPS = pd.concat([OPS, pd.DataFrame([{
    "target_precision": np.nan, "threshold": np.nan,
    "achieved_precision": BAND["precision"], "recall": np.nan,
    "share_of_pairs_flagged": BAND["coverage"]}])], ignore_index=True)
OPS["rule"] = list(["fitted gate"]*(len(OPS)-1)) + [BAND["rule"]]
print(OPS.to_string(index=False))
OPS.to_csv(os.path.join(OUT, "gate_operating_points.csv"), index=False)
print(f"\n'share_of_pairs_flagged' IS the deliverable size. Compare the fitted "
      f"rows against the band rule: if the band wins on coverage at similar "
      f"precision, ship the band — it is one line and auditable, and "
      f"explainability is a governance precondition for prospecting use.")

---
## 8. Error vs evidence — on a fixed cohort

**FIX 5.** In v1 the sample fell from 1.9M at k=1 to 10k at k=20, so every k
described a different population — high-degree targets are larger and more
national, and the apparent degradation past k=5 was composition change, not
evidence hurting.

Here the cohort is **frozen**: only targets with at least `K_MAX` located
neighbours enter, so the same nodes are scored at every k and the curve is
read as a within-target effect.

Three weighting schemes accumulate in one pass — uniform, amount (Block F's
convention, carried to be falsified), and inverse-variance. `tightest_first`
selects the narrowest kernels instead of drawing at random; in v1 it failed
to separate because the `bw = 0` degeneracy sorted to the front, so it is
retested here on guarded bandwidths only.

In [ ]:
KB = L.filter(F.col("kc_lat").isNotNull() & F.col("kc_bw_km").isNotNull())
cnt = KB.groupBy("a").agg(F.count("*").alias("n_nb"))
COHORT = cnt.filter(F.col("n_nb") >= K_MAX).select("a")
n_cohort = COHORT.count()
print(f"fixed cohort: {n_cohort:,} targets with >= {K_MAX} bandwidth-valid "
      f"located neighbours")
if n_cohort < 1000:
    print("WARNING: cohort too small — lower K_MAX or relax the bandwidth "
          "guard for this section only.")

K = (KB.join(COHORT, "a")
       .select("a", "b", "kc_lat", "kc_lon", "kc_bw_km", "amt", "a_lat", "a_lon"))
K = (unit_cols(K, "kc_lat", "kc_lon", pre="k")
       .withColumn("w_uni", F.lit(1.0))
       .withColumn("w_amt", F.col("amt"))
       .withColumn("w_ivr", F.lit(1.0) /
                   F.greatest(F.col("kc_bw_km"), F.lit(1.0))**2)).cache()

def prefix_curve(df, order_col, tag, reps):
    out = []
    for rep in range(reps):
        d = df.withColumn("_o", F.rand(SEED + rep) if order_col is None
                          else F.col(order_col))
        w = W.partitionBy("a").orderBy("_o")
        d = d.withColumn("k", F.row_number().over(w)).filter(F.col("k") <= K_MAX)
        wr = w.rowsBetween(W.unboundedPreceding, W.currentRow)
        for s in ("uni", "amt", "ivr"):
            for ax in ("x", "y", "z"):
                d = d.withColumn(f"{s}{ax}",
                                 F.sum(F.col(f"w_{s}")*F.col(f"ku{ax}")).over(wr))
        for s in ("uni", "amt", "ivr"):
            n = F.sqrt(F.col(f"{s}x")**2 + F.col(f"{s}y")**2 + F.col(f"{s}z")**2)
            d = (d.withColumn(f"{s}_lat", F.degrees(F.asin(F.col(f"{s}z")/n)))
                   .withColumn(f"{s}_lon", F.degrees(F.atan2(F.col(f"{s}y"),
                                                             F.col(f"{s}x"))))
                   .withColumn(f"e_{s}", gc_km(F.col("a_lat"), F.col("a_lon"),
                                               F.col(f"{s}_lat"),
                                               F.col(f"{s}_lon"))))
        g = (d.groupBy("k").agg(F.count("*").alias("n"),
                *[x for s in ("uni", "amt", "ivr") for x in
                  (F.expr(f"percentile_approx(e_{s},0.5)").alias(f"med_{s}"),
                   F.avg((F.col(f"e_{s}") <= 50).cast("double")).alias(f"hit50_{s}"))]))
        r = to_pd(g, f"{tag} rep{rep}"); r["rep"] = rep; r["order"] = tag
        out.append(r)
    return pd.concat(out, ignore_index=True)

CURVE = pd.concat([prefix_curve(K, None, "random", HARNESS_REPS),
                   prefix_curve(K, "kc_bw_km", "tightest_first", 1)],
                  ignore_index=True)
CURVE.to_csv(os.path.join(OUT, "error_vs_k.csv"), index=False)
CV = CURVE.groupby(["order", "k"]).mean(numeric_only=True).reset_index()
print(CV[CV.order == "random"][["k","n","med_uni","med_amt","med_ivr",
                                "hit50_ivr"]].to_string(index=False))
print(f"\n'n' must be ~constant across k now ({n_cohort:,}). If it still "
      f"falls, the cohort filter did not apply and the curve is confounded.")

In [ ]:
long = CV.melt(id_vars=["order", "k"],
               value_vars=["med_uni", "med_amt", "med_ivr"],
               var_name="scheme", value_name="median_err_km")
fig = px.line(long, x="k", y="median_err_km", color="scheme",
              line_dash="order", markers=True,
              title=f"Error vs located neighbours — FIXED COHORT of "
                    f"{n_cohort:,} targets ({VERSION}, {WINDOW})",
              color_discrete_sequence=PALETTE)
fig.add_hline(y=BASELINE["median_km"], line_dash="dot", line_color="crimson",
              annotation_text="footprint prior")
fig.update_layout(height=470, xaxis_title="k located neighbours used",
                  yaxis_title="median error km")
fig.show()

fig = px.line(CV, x="k", y="hit50_ivr", color="order", markers=True,
              title="Share of targets located within 50 km, by evidence count "
                    "(inverse-variance weighting)",
              color_discrete_sequence=PALETTE)
fig.update_layout(height=400, yaxis_tickformat=".0%",
                  xaxis_title="k located neighbours used")
fig.show()
print("On a fixed cohort the curve should fall monotonically and flatten. "
      "Where it flattens is where the marginal neighbour stops paying.")

### 8.1 The deliverable size is per NODE, not per pair

Everything so far is per-pair. The product is per-node: a target with ten
neighbours gets ten chances at a tight kernel, so the share of **targets** with
at least one narrow-kernel neighbour is far above the share of pairs. That
number — not the pair-level coverage — is what sizes the counterparty
population that can be located.

Selection is applied as the first run showed it should be: take the single
**narrowest** kernel per target rather than pooling.

In [ ]:
w1 = W.partitionBy("a").orderBy(F.col("kc_bw_km").asc())
BEST = (L.filter(F.col("kc_bw_km").isNotNull() & F.col("err_cnt_km").isNotNull())
          .withColumn("_r", F.row_number().over(w1)).filter(F.col("_r") == 1)
          .select("a", F.col("kc_bw_km").alias("best_bw"),
                  F.col("err_cnt_km").alias("best_err"), "b_naics2", "b_is_hub"))
n_tgt_any = BEST.count()
rows = []
for cut in [TIGHT_BW_KM, 10, 25, 50, 100, 250, 1e9]:
    r = (BEST.filter(F.col("best_bw") <= cut).agg(
            F.count("*").alias("n_targets"),
            F.expr("percentile_approx(best_err,0.5)").alias("median_km"),
            F.expr("percentile_approx(best_err,0.9)").alias("p90_km"),
            *[F.avg((F.col("best_err") <= x).cast("double")).alias(f"hit_{x}")
              for x in HIT_RADII_KM]).first().asDict())
    rows.append({"tightest_kernel_cut_km": cut,
                 "share_of_targets": r["n_targets"]/max(n_tgt_any,1), **r})
NODE = pd.DataFrame(rows)
print(f"targets with >=1 bandwidth-valid neighbour: {n_tgt_any:,}")
print(NODE.to_string(index=False))
NODE.to_csv(os.path.join(OUT, "node_level_coverage.csv"), index=False)
print(f"\nRead the TIGHT_BW_KM row as the product: that share of targets can be "
      f"located at that accuracy using ONE well-chosen neighbour. Compare it "
      f"against the pair-level band coverage ({BAND['coverage']:.1%}) — the "
      f"node-level figure is the honest deliverable size.")

### 8.2 Radius calibration — what the scoring function ships with

The estimator emits a **predicted** width (inverse-variance combination of the
selected kernels). To turn that into an honest radius we need the *empirical*
error distribution at each predicted width — not a Gaussian assumption, since
§5 showed the error distribution is heavy-tailed and partly bimodal.

This table is the calibration the scoring function reads: predicted width in,
`r50` / `r90` / `P(within 50 km)` out.

In [ ]:
def combo(df, k):
    """Tightest-k inverse-variance combination, per target."""
    d = (df.withColumn("_r", F.row_number().over(w1)).filter(F.col("_r") <= k)
           .withColumn("w", F.lit(1.0)/F.greatest(F.col("kc_bw_km"),
                                                  F.lit(1.0))**2))
    d = unit_cols(d, "kc_lat", "kc_lon", pre="k")
    g = (d.groupBy("a").agg(
            F.count("*").alias("k_used"),
            F.sum(F.col("w")*F.col("kux")).alias("vx"),
            F.sum(F.col("w")*F.col("kuy")).alias("vy"),
            F.sum(F.col("w")*F.col("kuz")).alias("vz"),
            F.sum("w").alias("sw"),
            F.first("a_lat").alias("a_lat"), F.first("a_lon").alias("a_lon")))
    # combined width of an inverse-variance mixture
    g = g.withColumn("pred_width_km", F.lit(1.0)/F.sqrt(F.col("sw")))
    g = vec_to_latlon(g, "vx", "vy", "vz", "est_lat", "est_lon")
    return (g.withColumn("err_km", gc_km(F.col("a_lat"), F.col("a_lon"),
                                         F.col("est_lat"), F.col("est_lon")))
             .withColumn("k_req", F.lit(k)))

BASE = L.filter(F.col("kc_bw_km").isNotNull() & F.col("err_cnt_km").isNotNull())
CAL = None
for k in (1, 3, 5):
    c = combo(BASE, k)
    CAL = c if CAL is None else CAL.unionByName(c)
CAL = CAL.cache()
CALB, edges = qbucket(CAL, "pred_width_km", 12)
RADCAL = to_pd(
    CALB.groupBy("bin").agg(
        F.count("*").alias("n"),
        F.min("pred_width_km").alias("width_lo"),
        F.max("pred_width_km").alias("width_hi"),
        F.expr("percentile_approx(err_km,0.5)").alias("r50_km"),
        F.expr("percentile_approx(err_km,0.9)").alias("r90_km"),
        F.avg((F.col("err_km") <= 50).cast("double")).alias("p_within_50km"),
        F.avg((F.col("err_km") <= 250).cast("double")).alias("p_within_250km"))
      .orderBy("bin"), "radius calibration")
print(RADCAL.to_string(index=False))
RADCAL.to_csv(os.path.join(OUT, "radius_calibration.csv"), index=False)
print("\nr50/r90 are EMPIRICAL quantiles of realised error at that predicted "
      "width, so they inherit the heavy tail instead of assuming it away. "
      "A parametric radius would be badly optimistic at the top end.")

fig = go.Figure()
fig.add_scatter(x=RADCAL.width_hi, y=RADCAL.r50_km, mode="lines+markers",
                name="realised r50", line=dict(width=3, color=PALETTE[0]))
fig.add_scatter(x=RADCAL.width_hi, y=RADCAL.r90_km, mode="lines+markers",
                name="realised r90", line=dict(width=3, color=PALETTE[1]))
fig.add_scatter(x=RADCAL.width_hi, y=RADCAL.width_hi, mode="lines",
                name="predicted width (y=x)", line=dict(dash="dash",
                                                        color="grey"))
fig.update_layout(height=460, xaxis_type="log", yaxis_type="log",
                  title="Predicted kernel width vs realised error radii — the "
                        "calibration the scoring function uses",
                  xaxis_title="predicted width km", yaxis_title="km")
fig.show()

---
## 9. The hub locating-power registry  *(read at `VERSION = "V0"`)*

The roadmap treats hubs as a flat exclusion list. The v1 run refuted that:
over half of real hubs carried usable location. A card processor has a
national cloud; a municipal utility, a school district or a parish has a cloud
the size of the geography it serves, and **a state-sized kernel is weak but
not useless** — at degree 1 it beats the national prior comfortably.

Two v1 caveats are now handled: hubs are defined by degree rather than by
ladder exclusion, and co-located clusters are gone, so a `PIN` class no longer
includes shared-coordinate artefacts.

In [ ]:
HUBP = (L.filter((F.col("b_is_hub") == 1) & F.col("err_cnt_km").isNotNull())
          .groupBy("b", "b_name", "b_naics2", "b_node_type", "b_deg")
          .agg(F.count("*").alias("n_located_cp"),
               F.avg("kc_bw_km").alias("bw_km"),
               F.avg("kc_bw_ok").alias("share_bw_trusted"),
               F.expr("percentile_approx(err_cnt_km,0.5)").alias("median_err_km"),
               F.avg((F.col("err_cnt_km") <= 250).cast("double")).alias("hit_250"),
               F.avg((F.col("err_cnt_km") <= 50).cast("double")).alias("hit_50"))
          .filter(F.col("n_located_cp") >= 50))
HP = to_pd(HUBP.orderBy("median_err_km"), "hub registry", max_rows=500_000)
HP["locating_class"] = pd.cut(HP.median_err_km, [-1, 25, 100, 400, 1e9],
                              labels=["PIN", "METRO", "REGION", "NONE"])
HP.to_csv(os.path.join(OUT, "hub_locating_registry.csv"), index=False)
print(HP.locating_class.value_counts().to_string())
print(f"\n{(HP.median_err_km <= 100).mean():.1%} of hubs locate to metro or "
      f"better — the case against a flat exclusion list.")
print("\n--- most informative hubs ---")
print(HP.head(20)[["b_name","b_naics2","b_deg","n_located_cp","bw_km",
                   "median_err_km","hit_250"]].to_string(index=False))
print("\n--- least informative (these are the true exclusions) ---")
print(HP.tail(10)[["b_name","b_naics2","b_deg","bw_km","median_err_km"]]
        .to_string(index=False))
print("\nSanity: PNC's own book-transfer accounts should appear in the NONE "
      "tail. If they sit in PIN, co-location leaked back in.")

In [ ]:
fig = px.scatter(HP, x="b_deg", y=HP.median_err_km.clip(lower=0.5),
                 color="locating_class", size="n_located_cp",
                 hover_data=["b_name", "b_naics2", "bw_km"],
                 log_x=True, log_y=True, color_discrete_sequence=PALETTE,
                 title="Hub locating power — degree does NOT determine whether "
                       "a hub carries location")
for r in HIT_RADII_KM:
    fig.add_hline(y=r, line_dash="dot", line_color="grey")
fig.update_layout(height=520, xaxis_title="hub degree",
                  yaxis_title="median degree-1 error km (clipped at 0.5)")
fig.show()

sec = (HP.groupby("b_naics2").agg(n=("b", "size"),
                                  median_err=("median_err_km", "median"))
         .query("n >= 5").sort_values("median_err"))
fig = px.bar(sec.reset_index(), x="b_naics2", y="median_err", hover_data=["n"],
             color_discrete_sequence=PALETTE,
             title="Hub sectors by locating power (utilities, education and "
                   "public administration should sit left)")
fig.update_layout(height=400, yaxis_title="median error km")
fig.show()
print(sec.to_string())

---
## 10. Footprint bias

Every observed neighbour is a PNC customer, so every prediction is pulled
toward the PNC footprint. Targets registered outside the core states are the
closest available analogue to a counterparty that banks elsewhere — which is
most of them. **This gap bounds the optimism in every number above.**

In [ ]:
core = [r[0] for r in (ATTR.filter("has_geo = 1").groupBy("state").count()
                       .orderBy(F.desc("count")).limit(6).collect())]
print(f"core footprint states: {core}")
ST = (LG.withColumn("in_core", F.col("a_state").isin(core))
        .groupBy("in_core").agg(
            F.count("*").alias("n"),
            F.expr("percentile_approx(err_cnt_km,0.5)").alias("median_km"),
            F.avg((F.col("err_cnt_km") <= 50).cast("double")).alias("hit_50")))
OOF = to_pd(ST, "in/out of footprint")
print(OOF.to_string(index=False))
if len(OOF) == 2:
    a, b = OOF.sort_values("in_core").median_km.to_list()
    print(f"\nout-of-core is {a/max(b,1e-9):.2f}x worse. That multiple is the "
          f"amount by which customer-derived accuracy OVERSTATES what to "
          f"expect on real counterparties. Quote it with every figure.")
OOF.to_csv(os.path.join(OUT, "footprint_bias.csv"), index=False)

In [ ]:
summary = {
    "version": VERSION, "window": WINDOW, "months": len(WIN_MONTHS),
    "pairs": int(n_pairs), "hub_nodes": int(n_hub),
    "coloc_excluded": bool(EXCLUDE_COLOCATED), "coloc_nodes": int(n_coloc),
    "cohort_targets": int(n_cohort),
    "prior_median_km": float(BASELINE["median_km"]),
    "prior_hit_50": float(BASELINE["hit_50"]),
    "share_no_bandwidth": float(aud["sh_no_bw"]),
    "gate_auc": float(auc), "gate_auc_bw_only": float(auc1),
    "band_rule": BAND["rule"], "band_precision": BAND["precision"],
    "band_coverage": BAND["coverage"], "tight_bw_km": float(TIGHT_BW_KM),
    "targets_with_valid_kernel": int(n_tgt_any),
    "node_share_tight": float(NODE.share_of_targets.iloc[0]),
    "node_median_km_tight": float(NODE.median_km.iloc[0]),
    "node_hit50_tight": float(NODE.hit_50.iloc[0]),
}
for _, r in TRUTH.iterrows():
    k = r["truth_set"].split(" (")[0].replace(" ", "_")
    summary[f"d1_{k}_n"] = int(r["n"])
    summary[f"d1_{k}_median_km"] = float(r["median_km"])
    summary[f"d1_{k}_hit_50"] = float(r["hit_50"])
for k in (1, 3, 5, 10, K_MAX):
    row = CV[(CV.order == "random") & (CV.k == k)]
    if len(row):
        summary[f"k{k}_median_ivr_km"] = float(row.med_ivr.iloc[0])
        summary[f"k{k}_hit50_ivr"] = float(row.hit50_ivr.iloc[0])
with open(os.path.join(OUT, "summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

---
## 11. Comparing configurations

Run all four — `V0`/`P99_9` × `last3`/`all` — then this section reads the
saved summaries.

- **`last3` vs `all`.** Pooling 23 months adds edges and stales the address
  attribution. The difference is the **net**, not the staleness cost alone.
- **`V0` vs `P99_9`.** Whether hubs help in aggregate. This can be negative
  overall while §9 shows specific hubs are excellent — which is the argument
  for a graded registry rather than a flat exclusion.

In [ ]:
rows = []
for d in sorted(os.listdir(OUT_ROOT)):
    p = os.path.join(OUT_ROOT, d, "summary.json")
    if os.path.exists(p): rows.append(json.load(open(p)))
if not rows:
    print("no configurations written yet")
else:
    CMP = pd.DataFrame(rows).sort_values(["version", "window"])
    keep = [c for c in CMP.columns if c in ("version","window","pairs")
            or "hit_50" in c or "median_km" in c or c.startswith("k")]
    print(CMP[keep].to_string(index=False))
    CMP.to_csv(os.path.join(OUT_ROOT, "config_comparison.csv"), index=False)
    if len(CMP) > 1:
        m = CMP.melt(id_vars=["version", "window"],
                     value_vars=[c for c in CMP.columns if "hit_50" in c
                                 or "hit50" in c],
                     var_name="metric", value_name="value")
        fig = px.bar(m, x="metric", y="value", color="window",
                     facet_row="version", barmode="group",
                     color_discrete_sequence=PALETTE,
                     title="Hit rates across configurations — window "
                           "(staleness vs volume) and rung (hubs in / out)")
        fig.update_layout(height=560, yaxis_tickformat=".0%")
        fig.show()

---
## 12. Reading the results

**The three numbers that decide the programme.**

1. `d1_pin_representative_hit_50` — accuracy where the ground truth is
   trustworthy. In v1 this was **51%** within 50 km from a *single* neighbour,
   against a footprint prior of 0.014%. This is the real capability.
2. `share_of_pairs_flagged` at the chosen precision, **or** `band_coverage` if
   the one-line rule wins. This is the deliverable size: if it is 15%, then 15%
   of counterparties get a CBSA and 85% get an honest `unknown` — a usable
   asset. A point estimate for 100% is not.
3. The out-of-core multiple — how much customer-derived accuracy overstates
   what counterparties will do.

**What each section settles.**

| Result | Consequence |
|---|---|
| binned-bandwidth-only AUC ≈ full AUC | ship the one-feature gate; auditable line by line |
| band rule ≥ fitted gate on coverage | ship the interval, not the model |
| sector effect vanishes conditional on bandwidth | NAICS belongs in the **prior on kernel width**, not its own term |
| `med_amt` > `med_uni` | dollars do not carry location |
| inverse-variance ≪ uniform | weighting by kernel precision is the highest-value design choice — in v1, 4× better at k=3 |
| fixed-cohort curve flattens by k≈4–5 | the marginal neighbour stops paying early; effort goes to the gate |
| hubs split PIN/METRO/REGION/NONE | replace the flat exclusion list with the graded registry |

**What this cannot tell you.**

- **Distribution shift is real and one-directional.** Counterparties bank
  elsewhere and their observed neighbour sets are a more biased sample of
  their true counterparty base. Every number here is **optimistic**; the
  out-of-core split bounds it but does not remove it.
- **`scope = on_us_c2c`.** A counterparty's real cloud is mostly invisible.
- **No external ground truth.** The FI pinning registry (FDIC Summary of
  Deposits, NCUA) is the only available set of counterparties with known
  locations — small and skewed to financial institutions, but drawn from the
  *right population*. A precondition for briefing accuracy outside the team.
- **Co-location is excluded, not solved.** Those parties still need locating;
  they simply cannot serve as evidence or as truth. Sizing that population is
  a deliverable in its own right.

**Sequencing.**

1. Run all four configurations; settle window and rung before tuning.
2. Freeze the gate at a precision target. Emit **`(cell, radius, p)` — never a
   bare lat/lon.**
3. Validate against the FI registry; report the customer-to-counterparty gap.
4. Only then consider learned models. This estimator is one round of message
   passing with hand-set weights, which makes it the baseline any GNN must
   beat. A GNN will not help at degree 1 — architecture does not create data —
   but 2-hop co-location through shared customers is genuine headroom. If
   pursued, predict over a **discrete cell grid**: it yields a real posterior
   and handles multimodality, which a Gaussian head cannot.

---
## 13. `locate()` — the scoring function

Everything above is measurement. This is the thing you call.

**Two modes, one estimator.**

| mode | call | use |
|---|---|---|
| by node id | `locate(ids=["1000..."])` | a node already in the graph — neighbours are looked up |
| by neighbour list | `locate(neighbours={"CPTY_1": ["1000...", "1000..."]})` | an **external counterparty**, which has no `mdm_id` in the metric table but whose PNC counterparties are known |

The second mode is the one that matters when PAYS_CPTY lands: pass the
customers a counterparty transacts with, get a location back.

**What it returns, and what it deliberately does not.** It returns
`(est_lat, est_lon, pred_width_km, r50_km, r90_km, p_within_50km, ...)` and an
`evidence_class`. It never returns a bare coordinate: a point with no radius
is the thing that cannot be used responsibly downstream, because nothing about
it distinguishes a 15 km estimate from a 900 km one.

**Design choices carried in from the analysis.**

- **Tightest-*k* selection, not pooling.** §8 showed selecting the narrowest
  kernels beats averaging *k* random ones outright — uniform averaging is flat
  at ~190 km from k=1 to k=20, because one national counterparty drags a good
  local one to the middle of nowhere.
- **Inverse-variance weighting** among the selected kernels.
- **Leave-one-out when the target is itself located** (a customer sits inside
  its neighbours' clouds; an external counterparty does not). `exclude_self`
  handles both cases, and getting it wrong is silent circularity.
- **Co-located and unguarded neighbours are dropped**, matching the harness.
- **Radii come from `radius_calibration.csv`** — empirical quantiles of
  realised error at that predicted width, not a Gaussian assumption.

In [ ]:
RADCAL_PATH = os.path.join(OUT, "radius_calibration.csv")

def _load_radcal(path=RADCAL_PATH):
    r = pd.read_csv(path).sort_values("width_hi").reset_index(drop=True)
    return r

def _calibrate(pred_width, radcal):
    """Map a predicted width to empirical r50 / r90 / P(within 50 km).

    Nearest calibration bin by upper width edge. Anything beyond the last bin
    inherits the last bin's radii — deliberately pessimistic, because the tail
    is where a parametric extrapolation would be most wrong.
    """
    idx = np.searchsorted(radcal.width_hi.to_numpy(), pred_width, side="left")
    idx = np.clip(idx, 0, len(radcal) - 1)
    r = radcal.iloc[idx]
    return (r.r50_km.to_numpy(), r.r90_km.to_numpy(),
            r.p_within_50km.to_numpy(), r.p_within_250km.to_numpy())

def locate(ids=None, neighbours=None, k=5, exclude_self=None,
           min_bw_km=None, max_bw_km=None, radcal=None):
    """Estimate location + radius for nodes or for external counterparties.

    ids        : iterable of mdm_id already in the graph; neighbours are
                 looked up from the pooled pair table.
    neighbours : dict {target_key: [neighbour mdm_id, ...]} for entities NOT
                 in the graph (the counterparty case).
    k          : how many of the narrowest kernels to combine.
    exclude_self : leave-one-out. Defaults to True for `ids` (a customer is
                 inside its own neighbours' clouds — not subtracting it is
                 circular) and False for `neighbours` (an external
                 counterparty is not in any located cloud).
    max_bw_km  : drop kernels wider than this before selecting. Defaults to
                 TIGHT_BW_KM; set None to keep all.
    """
    radcal = _load_radcal() if radcal is None else radcal
    max_bw_km = TIGHT_BW_KM if max_bw_km == "tight" else max_bw_km

    if ids is not None:
        exclude_self = True if exclude_self is None else exclude_self
        tgt = spark.createDataFrame(
            pd.DataFrame({"a": [str(x) for x in ids]}))
        E2 = PAIR.join(tgt, "a").select("a", "b", "amt")
    elif neighbours is not None:
        exclude_self = False if exclude_self is None else exclude_self
        rows = [(str(t), str(n)) for t, ns in neighbours.items() for n in ns]
        E2 = (spark.createDataFrame(pd.DataFrame(rows, columns=["a", "b"]))
                   .withColumn("amt", F.lit(0.0)))
    else:
        raise ValueError("pass ids= or neighbours=")

    # neighbour clouds, then LOO only where the target is genuinely inside one
    Q = E2.join(NB, "b")
    if exclude_self:
        Q = (Q.join(LOC.select(F.col("node").alias("a"), "ux", "uy", "uz"),
                    "a", "left")
               .withColumn("w1", F.when(F.col("ux").isNotNull(),
                                        F.lit(1.0)).otherwise(F.lit(0.0)))
               .fillna({"ux": 0.0, "uy": 0.0, "uz": 0.0}))
    else:
        Q = (Q.withColumn("ux", F.lit(0.0)).withColumn("uy", F.lit(0.0))
               .withColumn("uz", F.lit(0.0)).withColumn("w1", F.lit(0.0)))
    Q = (Q.withColumn("n_cp_loo", F.col("n_cp_loc") - F.col("w1").cast("int"))
           .transform(lambda d: loo_block(d, "kc_", "n_cp_loc", "cV_x", "cV_y",
                                          "cV_z", "w1", "n_cp_loo")))
    # same guards as the harness: real kernel, not co-located, width in range
    Q = Q.filter(F.col("kc_bw_km").isNotNull() & F.col("kc_lat").isNotNull())
    if min_bw_km is not None: Q = Q.filter(F.col("kc_bw_km") >= min_bw_km)
    if max_bw_km is not None: Q = Q.filter(F.col("kc_bw_km") <= max_bw_km)

    wsel = W.partitionBy("a").orderBy(F.col("kc_bw_km").asc())
    Q = (Q.withColumn("_r", F.row_number().over(wsel)).filter(F.col("_r") <= k)
           .withColumn("w", F.lit(1.0)/F.greatest(F.col("kc_bw_km"),
                                                  F.lit(1.0))**2))
    Q = unit_cols(Q, "kc_lat", "kc_lon", pre="k")
    G2 = (Q.groupBy("a").agg(
            F.count("*").alias("k_used"),
            F.min("kc_bw_km").alias("tightest_bw_km"),
            F.sum(F.col("w")*F.col("kux")).alias("vx"),
            F.sum(F.col("w")*F.col("kuy")).alias("vy"),
            F.sum(F.col("w")*F.col("kuz")).alias("vz"),
            F.sum("w").alias("sw"))
           .withColumn("pred_width_km", F.lit(1.0)/F.sqrt(F.col("sw"))))
    G2 = vec_to_latlon(G2, "vx", "vy", "vz", "est_lat", "est_lon")
    out = G2.select("a", "k_used", "tightest_bw_km", "pred_width_km",
                    "est_lat", "est_lon").toPandas()

    # everything asked for, including targets with no usable evidence
    keys = ([str(x) for x in ids] if ids is not None
            else [str(t) for t in neighbours])
    out = pd.DataFrame({"a": keys}).merge(out, on="a", how="left")
    ok = out.pred_width_km.notna()
    for c in ("r50_km", "r90_km", "p_within_50km", "p_within_250km"):
        out[c] = np.nan
    if ok.any():
        r50, r90, p50, p250 = _calibrate(out.loc[ok, "pred_width_km"].to_numpy(),
                                         radcal)
        out.loc[ok, ["r50_km", "r90_km", "p_within_50km",
                     "p_within_250km"]] = np.column_stack([r50, r90, p50, p250])
    out["evidence_class"] = np.select(
        [~ok,
         out.tightest_bw_km <= (TIGHT_BW_KM or 25),
         out.r50_km <= 100, out.r50_km <= 400],
        ["NO_EVIDENCE", "PIN", "METRO", "REGION"], "NONE")
    return out.rename(columns={"a": "target"})[
        ["target", "evidence_class", "est_lat", "est_lon", "k_used",
         "tightest_bw_km", "pred_width_km", "r50_km", "r90_km",
         "p_within_50km", "p_within_250km"]]
print("locate() ready")

### 13.1 Smoke test and honest self-scoring

Two checks. First that both call modes return sane objects. Second — the one
that matters — that scoring *known* customers through the same function
reproduces the harness numbers. If `locate()` looks better than §8 on the same
nodes, it is leaking: almost certainly `exclude_self` not applying.

In [ ]:
# --- mode 1: nodes already in the graph -----------------------------------
sample_ids = [r[0] for r in BEST.select("a").limit(500).collect()]
R1 = locate(ids=sample_ids[:200], k=5, max_bw_km="tight")
print(R1.head(10).to_string(index=False))
print("\nevidence_class mix:")
print(R1.evidence_class.value_counts().to_string())

# --- mode 2: the counterparty case, target NOT in the graph ---------------
demo = {}
for a in sample_ids[:5]:
    nbrs = [r[0] for r in PAIR.filter(F.col("a") == a).select("b").limit(8).collect()]
    demo[f"CPTY_{a}"] = nbrs
R2 = locate(neighbours=demo, k=5, max_bw_km="tight")
print("\ncounterparty-style call (no mdm_id for the target):")
print(R2.to_string(index=False))
print("\nNote exclude_self defaults FALSE here: an external counterparty is "
      "not inside any located cloud, so there is nothing to subtract. Passing "
      "a customer id through this mode would silently leak the answer.")

In [ ]:
# --- self-scoring: does locate() reproduce the harness? -------------------
truth = (ATTR.filter("has_geo = 1").select(F.col("node").alias("target"),
         F.col("lat").alias("true_lat"), F.col("lon").alias("true_lon"))
         .join(spark.createDataFrame(pd.DataFrame({"target": sample_ids})),
               "target").toPandas())
chk = locate(ids=list(truth.target), k=5, max_bw_km="tight").merge(truth, on="target")
chk = chk[chk.est_lat.notna()].copy()
if len(chk):
    la1, lo1 = np.radians(chk.true_lat), np.radians(chk.est_lat)
    dl = np.radians(chk.est_lon - chk.true_lon)
    aa = (np.sin((lo1-la1)/2)**2 + np.cos(la1)*np.cos(lo1)*np.sin(dl/2)**2)
    chk["err_km"] = 2*R_EARTH_KM*np.arcsin(np.sqrt(np.clip(aa, 0, 1)))
    print(f"self-scored {len(chk):,} known customers through locate():")
    print(f"  median err {chk.err_km.median():,.1f} km | "
          f"hit@50 {(chk.err_km <= 50).mean():.1%}")
    print(f"  realised within r50: {(chk.err_km <= chk.r50_km).mean():.1%} "
          f"(should be ~50%)")
    print(f"  realised within r90: {(chk.err_km <= chk.r90_km).mean():.1%} "
          f"(should be ~90%)")
    print("\nIf the r50/r90 coverage is far from 50/90, the calibration table "
          "does not transfer to this slice and must be refitted before the "
          "function is used on counterparties.")
    print("If the median error is much BETTER than §8 on comparable k, "
          "exclude_self is not applying and the estimate is circular.")

### 13.2 Using it on real counterparties

```python
# one counterparty, from the customers it transacts with
locate(neighbours={"CPTY_ACME": ["100055096284", "100066477143"]})

# batch, straight from a PAYS_CPTY pull
cp = (spark.table("<pays_cpty>")
        .groupBy("cpty_id").agg(F.collect_set("mdm_id").alias("nbrs"))
        .toPandas())
locate(neighbours=dict(zip(cp.cpty_id, cp.nbrs)), k=5, max_bw_km="tight")
```

**Read the output as a gate, not a pin.** `evidence_class` is the field to
route on; `est_lat`/`est_lon` are only meaningful alongside `r50_km`.

Three cautions that do not go away by using the function:

- **Counterparties are not customers.** The out-of-footprint split measures how
  much this overstates on entities that bank elsewhere. Apply that multiple to
  every radius before briefing, or refit the calibration on the FI pinning
  registry once it exists.
- **`NO_EVIDENCE` is a real answer.** Do not backfill it with a footprint
  centroid. A node with no usable kernel is honestly unlocated, and the whole
  point of the gate is that the two cases stay distinguishable.
- **`max_bw_km="tight"` is the default posture.** Dropping it raises coverage
  and degrades precision; that trade is the `node_level_coverage.csv` table,
  and it is a product decision rather than a modelling one.